# 19 — Can uncertainty warnings help us review tags?

**DR-62 · Gemini 3.8 Flash · retrospective development diagnostic**

We ask a practical question: can the model flag answers that need attention, including missing tags? We reuse 54 reviewed development cases and their original saved answers. This is a second assessment call, so it does not tell us what the model knew during its first answer. Reference labels are used only for scoring and are never sent to the provider.

The notebook runs in replay mode by default. Private files are required to reproduce the results; public findings are in `docs/results/confidence-diagnostic.md`. No new labeling task is required.

In [ ]:
import json
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path[:0] = [str(root), str(root / "src")]
from dotenv import load_dotenv
from notebooks.tools import show_table, fingerprint, verify_sha256
from notebooks.tools.confidence import decode_confidence
from notebooks.tools.gem_profile import gem_request
from notebooks.tools.gemini import decode_result
from notebooks.tools.costs import BudgetGuard
from demo_logic.notebook_support import save_private_json
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags

load_dotenv(root / ".env", override=False)
LIVE = os.getenv("NOTEBOOK19_RUN_LIVE", "false").lower() == "true"
STAGE = os.getenv("NOTEBOOK19_STAGE", "audit")
if STAGE not in {"audit", "search"}:
    raise ValueError("NOTEBOOK19_STAGE must be audit or search")
folder = root / "reports/confidence-review/v1"
config = json.loads((root / "notebooks/prompts/confidence-diagnostic-v1.json").read_text())
tags = sorted(topic_tags("v2"))
key = os.getenv("GEMINI_API_KEY", "")
guard = BudgetGuard(root / "reports/budget/capstone.sqlite", prior_total_microusd=0,
                    prior_research_microusd=0, run_limits={"confidence-diagnostic-v1": 3_000_000})

## 1. Keep the answers and reference fixed

The 54 cases are accepted DR-61 references. Three unresolved cases stay out. We keep the complete reference labels, even when the old prediction could return only two tags. A disagreement means the tag sets differ; it is not automatically proof of a model mistake.

In [ ]:
sources = {
    "reports/error-analysis/2026-10-05/approved-reference-v1.json": "b508d0a3b9b803f7d018816c753e1673860c1a297acd8733607bf6c1f67fe6fe",
    "reports/gemini-broader/v1/development-results.json": "59849e7e463a30553454c0ff16ed9fc50f0e4e13c6d5d1f134f9de14e008490b",
    "data/dataset.jsonl": "81ba27a52fb25a7fc1ae0a256522a9c22c39c97dfeac1b8cbbecd2e956d67ca4",
    "docs/tagging-rubric.md": config["rubric_sha256"],
}
for path, expected in sources.items():
    verify_sha256(root / path, expected)
reference = json.loads((root / list(sources)[0]).read_text())
records = [r for r in reference["records"] if r["score_eligible"]]
assert len(records) == 54
saved = json.loads((root / list(sources)[1]).read_text())["answers"][config["model"]]
baseline = {r["external_id"]: r for r in saved}
assert all(baseline[r["external_id"]]["status"] == "ok" for r in records)
contract = {"config": config, "source_sha256": sources, "ids": sorted(r["external_id"] for r in records)}
save_private_json(folder / "contract.json", contract)
show_table(["Input", "Count"], [("Accepted development cases", len(records)), ("Final-test cases", 0)])

## 2. Ask about support and missing topics

**Support** asks: does the title justify the suggested tags? **Completeness** asks: did the answer leave out an important topic?

We flag medium or low on either question. High is a self-report, not a measured probability. A vague event name may need search; a disagreement about the rubric needs human judgment instead. The full fixed prompt is shown below.

In [ ]:
print(config["audit_system"])
audits = {}
for row in records:
    event_id = row["external_id"]
    payload = {"model": config["model"], "system_instruction": config["audit_system"],
               "input": json.dumps({"title": row["title"], "suggested_tags": baseline[event_id]["predicted"]}, ensure_ascii=False),
               "generation_config": config["generation_config"], "store": False}
    audits[event_id] = gem_request(payload, event_id=event_id, folder=folder / "audit", key=key,
                                  tags=tags, guard=guard, live=LIVE, decoder=decode_confidence,
                                  run_id="confidence-diagnostic-v1")
    print(f"Audit {len(audits)}/{len(records)}: {audits[event_id]['status']}")
    if audits[event_id].get("accounting_pending") or audits[event_id]["status"] in {"budget_paused", "rate_limit_paused", "http_error", "network_or_decode_error"}:
        break

## 3. Do warnings find disagreements?

We count warnings that match a reference disagreement, disagreements missed by the warning, and warnings on exact matches. These last warnings may still be reasonable: matching a reference does not guarantee the title was clear.

Before running the model, we set a simple exploratory gate for search: warnings must catch at least half of disagreements, flag fewer than 80% of cases, and be more frequent among disagreements than exact matches. There must also be missing-context cases. These are trial rules, not established quality standards. Failed responses stay visible and cannot pass the gate.

In [ ]:
complete = len(audits) == len(records) and all(a["status"] == "ok" for a in audits.values())
rows = []
for r in records:
    a = audits.get(r["external_id"], {})
    if a.get("status") != "ok":
        continue
    audit = a["audit"]
    rows.append({"external_id": r["external_id"], "flagged": audit["support"] != "high" or audit["completeness"] != "high",
                 "disagreement": set(baseline[r["external_id"]]["predicted"]) != set(r["reference_tags"]),
                 "cause": audit["cause"], "reason": audit["reason"]})
caught = sum(r["flagged"] and r["disagreement"] for r in rows)
missed = sum(not r["flagged"] and r["disagreement"] for r in rows)
false_warnings = sum(r["flagged"] and not r["disagreement"] for r in rows)
quiet_matches = sum(not r["flagged"] and not r["disagreement"] for r in rows)
flagged = caught + false_warnings
quiet = missed + quiet_matches
missing_context = [r["external_id"] for r in rows if r["flagged"] and r["cause"] == "missing_context"]
gate = (complete and caught + missed > 0 and 0 < flagged < .8 * len(records) and quiet > 0
        and caught / (caught + missed) >= .5 and caught / flagged > missed / quiet and bool(missing_context))
warning_summary = {"complete": complete, "disagreements_caught": caught, "disagreements_missed": missed,
                   "warnings_on_matches": false_warnings, "unflagged_matches": quiet_matches,
                   "flagged": flagged, "missing_context": len(missing_context), "search_gate_passed": gate}
show_table(["Measure", "Count / result"], list(warning_summary.items()))
if complete:
    save_private_json(folder / "warning-results.json", {"summary": warning_summary, "rows": rows})

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

if complete:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    labels = ["Disagreements\ncaught", "Disagreements\nmissed", "Warnings on\nmatching answers", "Matching answers\nnot flagged"]
    values = [caught, missed, false_warnings, quiet_matches]
    bars = ax.bar(labels, values, color=["#218380", "#db504a", "#e9b44c", "#536878"])
    ax.bar_label(bars)
    ax.set_ylabel("Development cases")
    ax.set_title("Does the warning identify answers needing attention?")
    ax.set_ylim(0, max(values) + 3)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


## 4. Search only where missing context was flagged

If the gate passes, take up to eight flagged missing-context cases in a fixed hash order. Reference answers do not choose this subset. Both new tagging calls receive the same title, date and location; only search availability changes. This keeps added event context from being confused with a search benefit. Compare these two new answers with each other, not with the older title-only answer.

Set `NOTEBOOK19_STAGE=search` and `NOTEBOOK19_RUN_LIVE=true` only for this stage. Source citations and actual search traces are saved privately. Search being enabled does not prove it ran or found the correct event.

In [ ]:
selected = sorted(missing_context, key=lambda x: fingerprint(x))[:config["search_max_cases"]] if gate else []
if complete:
    save_private_json(folder / "search-selection.json", {"gate_passed": gate, "selected_ids": selected,
                      "selection_rule": config["selection"]})
snapshot = {r["external_id"]: r for r in map(json.loads, (root / "data/dataset.jsonl").read_text().splitlines())}
paired = {}
for event_id in selected:
    r = snapshot[event_id]
    paired[event_id] = {}
    for search in (False, True):
        system = config["tagging_system"].replace("Use only the supplied title.", "Use the supplied event fields. When search is available, verify the same event and date before using external facts. Do not transfer demands from another edition. If no reliable match is found, use the title only.")
        payload = {"model": config["model"], "system_instruction": system,
                   "input": json.dumps({k: r.get(k) for k in ("title", "date", "postcode", "venue_tokens")}, ensure_ascii=False),
                   "generation_config": config["generation_config"], "store": False}
        if search:
            payload["tools"] = [{"type": "google_search"}]
        paired[event_id][str(search)] = gem_request(payload, event_id=event_id, folder=folder / "search",
                key=key, tags=tags, guard=guard, live=LIVE and STAGE == "search", decoder=decode_result,
                run_id="confidence-diagnostic-v1")
        print(f"Search={search}: {paired[event_id][str(search)]['status']}")
        if paired[event_id][str(search)].get("accounting_pending"):
            raise RuntimeError("Reconcile usage before continuing")

## 5. Compare the paired answers and decide the next step

An improvement means fewer missing plus extra tags against the same reference. Report regressions too. This selected subgroup cannot establish a general search benefit. Inspect the private source traces for wrong dates, nearby events and unsupported claims before recommending automatic search.

After this step, discuss the results and next steps with the project owner. DR-24 accounting and DR-27's service decision remain open; this notebook does not authorize deployment.

In [ ]:
paired_complete = bool(selected) and all(a["status"] == "ok" for pair in paired.values() for a in pair.values())
if paired_complete:
    refs = {r["external_id"]: r["reference_tags"] for r in records}
    comparisons = []
    for event_id in selected:
        truth = set(refs[event_id])
        off = set(paired[event_id]["False"]["predicted"])
        on = set(paired[event_id]["True"]["predicted"])
        change = len(on ^ truth) - len(off ^ truth)
        comparisons.append({"external_id": event_id, "off_errors": len(off ^ truth), "on_errors": len(on ^ truth),
                            "change": "improved" if change < 0 else "worse" if change > 0 else "unchanged",
                            "search_executed": paired[event_id]["True"].get("search_executed", False)})
    metrics = {mode: multilabel_metrics([refs[i] for i in selected], [paired[i][mode]["predicted"] for i in selected], tags) for mode in ("False", "True")}
    save_private_json(folder / "paired-results.json", {"comparisons": comparisons, "metrics": metrics})
    show_table(["Mode", "Precision", "Recall", "Coverage"], [(mode, m["micro_precision"], m["micro_recall"], m["coverage"]) for mode,m in metrics.items()])
elif complete and not gate:
    print("The warning rule did not pass the search gate. Do not run targeted search from this signal.")
else:
    print("Results are incomplete. Finish the selected stage before drawing a conclusion.")

## Recorded finding — 6 October

**Warnings caught 8 of 27 disagreements and missed 19.** Ten of 54 answers were flagged; two of those already matched the reference. Among the 44 unflagged answers, 19 still differed from the reference.

The warnings identified some cases worth reviewing, but high certainty was not a dependable sign of a complete answer. The search gate failed: no targeted-search calls were made. Four missing-context cases could support a separately agreed exploratory search experiment, but they do not change this gate's result.

All 54 assessments succeeded. The conservative usage estimate was **$0.565635** and median additional request latency **6.82 seconds**. The original tags, reference labels and their quality scores remain unchanged.

**Next checkpoint:** discuss whether to finish accounting and recommend a reviewed prototype, or define a separate small context experiment first. DR-24 and DR-27 stay open. No extra labeling task is requested. See the [full aggregate summary](../docs/results/confidence-diagnostic.md).


## 6. Explain the eight answers with no shared tags

This follow-up reruns the eight cases whose old Gemini tags had no overlap with the accepted reference, including four abstentions. The model receives the original title and same rubric excerpt, with search off. It does **not** receive the reviewed tags or its old prediction.

The output now includes a short explanation for each selected tag, a reason for choosing no tags, and any specific rubric ambiguity. That instruction change may influence the answer. Compare old and new tags before using the explanation to understand the original disagreement; the original calls did not save reasons.

Default execution replays saved answers. Set `NOTEBOOK19_EXPLAIN_LIVE=true` only to finish this eight-case run. No reference labels are changed. Its $1 run ceiling remains within the shared research and per-event limits.

In [ ]:
from notebooks.tools.hosted import parse_llm_tags

explain_config = json.loads((root / "notebooks/prompts/decision-explanation-v1.json").read_text())
EXPLAIN_LIVE = os.getenv("NOTEBOOK19_EXPLAIN_LIVE", "false").lower() == "true"
explain_folder = folder / "decision-explanation-v1"
explain_guard = BudgetGuard(root / "reports/budget/capstone.sqlite", prior_total_microusd=0,
                           prior_research_microusd=0, run_limits={"decision-explanation-v1": 1_000_000})
explain_cases = [r for r in records if not set(r["reference_tags"]) & set(baseline[r["external_id"]]["predicted"])]
assert len(explain_cases) == 8
save_private_json(explain_folder / "contract.json", {"config": explain_config,
    "source_sha256": sources, "selected_ids": [r["external_id"] for r in explain_cases]})


def decode_decision(body, allowed):
    """Validate concise decision explanations while preserving the two-tag policy.

    Invalid schemas or missing abstention reasons are failures, not empty answers.
    This is a reported explanation of a new response, not original model reasoning.
    """
    def parse(content, vocabulary):
        obj = json.loads(content)
        if not isinstance(obj, dict) or set(obj) != {"tags", "reason", "abstention_reason", "rubric_ambiguity"}:
            raise ValueError("Unexpected decision fields")
        parse_llm_tags(json.dumps({"tags": obj["tags"]}), vocabulary)
        if any(not isinstance(obj[k], str) for k in ("reason", "abstention_reason", "rubric_ambiguity")):
            raise ValueError("Explanations must be strings")
        if obj["tags"] and (not obj["reason"].strip() or obj["abstention_reason"].strip()):
            raise ValueError("Chosen tags need a reason and no abstention reason")
        if not obj["tags"] and not obj["abstention_reason"].strip():
            raise ValueError("Empty tags need an abstention reason")
        return obj
    decoded = decode_result(body, allowed, content_parser=parse)
    decision = decoded.pop("predicted")
    return {**decoded, "predicted": decision["tags"], "decision": decision}


explanations = {}
for r in explain_cases:
    payload = {"model": explain_config["model"], "system_instruction": explain_config["system_instruction"],
               "input": json.dumps({"title": r["title"]}, ensure_ascii=False),
               "generation_config": explain_config["generation_config"], "store": False}
    result = gem_request(payload, event_id=r["external_id"], folder=explain_folder, key=key,
                         tags=tags, guard=explain_guard, live=EXPLAIN_LIVE, decoder=decode_decision,
                         run_id="decision-explanation-v1")
    explanations[r["external_id"]] = result
    print(f"Explanation {len(explanations)}/8: {result['status']}")
    if result.get("accounting_pending") or result["status"] in {"budget_paused", "rate_limit_paused", "http_error", "network_or_decode_error"}:
        break


## 7. Read the decision explanations

The private table shows the title, accepted tags, old prediction and new prediction. When the tags change, the explanation belongs to that changed answer. These eight difficult cases can reveal unclear rules or missing information, but they cannot estimate general accuracy.

Review the specific rule cited by the model. Is it following our intended definition, choosing a secondary topic, or asking for facts absent from the title? Keep rubric changes separate from correcting individual reference labels.

In [ ]:
explain_complete = len(explanations) == 8 and all(x["status"] == "ok" for x in explanations.values())
if explain_complete:
    explain_rows = []
    for r in explain_cases:
        result = explanations[r["external_id"]]
        decision = result["decision"]
        old = baseline[r["external_id"]]["predicted"]
        explain_rows.append({"external_id": r["external_id"], "title": r["title"],
            "reference_tags": r["reference_tags"], "old_tags": old, "new_tags": result["predicted"],
            "prediction_changed": set(old) != set(result["predicted"]), **{k: decision[k] for k in ("reason", "abstention_reason", "rubric_ambiguity")}})
    explain_summary = {"records": 8, "prediction_changes": sum(x["prediction_changed"] for x in explain_rows),
        "gross_upper_microusd": sum(x["gross_upper_microusd"] for x in explanations.values()),
        "search_calls": 0, "dashboard_before_run_eur": "2.32"}
    save_private_json(explain_folder / "results.json", {"summary": explain_summary, "rows": explain_rows})
    show_table(["Title", "Reviewed tags", "Old Gemini", "New Gemini", "Decision explanation", "Rubric ambiguity"],
        [(x["title"], "; ".join(x["reference_tags"]), "; ".join(x["old_tags"]) or "None",
          "; ".join(x["new_tags"]) or "None", x["reason"] or x["abstention_reason"], x["rubric_ambiguity"])
         for x in explain_rows])
    print(explain_summary)
else:
    print("Finish the eight-case run before interpreting explanations.")


## Explanation follow-up: what we learned

**Seven of eight predictions stayed the same.** Sivas changed from `culture; religion` to `culture; international-politics`. All four original abstentions remained empty.

The model's explanations point to different issues: missing incident details, participation versus the event's subject, a narrow interpretation of German politics, and the distinction between protest format and topic. The stalking example also needs checking against the existing protection-from-violence rule. An explanation does not prove that the model applied a rule correctly.

We should discuss a few specific boundaries rather than assume the whole rubric is wrong. Keep original labels and scores fixed until a change is explicitly agreed. These are new response explanations, not records of the original decisions.

Eight calls succeeded, with no retries or search. Additional conservative usage estimate: **$0.104131**. The latest reported dashboard total before this run was **€2.32**. The follow-up does not close DR-24 or DR-27.
